# Govern an agent's MCP tool calls at the LLM gateway, with no MCP gateway

Your Strands agent calls its MCP servers directly. Only its model calls go through Highflame: to
the Highflame LLM gateway, with a Highflame token for the person who signed in. That is enough for
Highflame to see every tool call. The model asks for each tool call in its response, and the LLM
gateway checks that response before the agent gets it.

| What Highflame does | How |
| --- | --- |
| Records each tool call that the model asks for, for the person and the agent | The LLM gateway reads the tool calls in each model response. |
| Refuses a tool call before the agent runs it | A gateway policy that forbids the tool by its name. The MCP server never gets the call. |

The MCP server does not need to be registered in Highflame. This notebook uses
[`example_mcp_server.py`](example_mcp_server.py), a small MCP server that runs on your machine and
that Highflame does not know about.

> **Customer-managed deployments only.** Highflame links the Okta person to a member of your
> organization by a verified email on one of your claimed domains. That needs a deployment
> with domain join. Highflame's hosted service does not offer it yet.

| Part | What it shows |
| --- | --- |
| A | The person's token for the LLM gateway: sign-in, then the exchange |
| B | The MCP server, outside Highflame |
| C | The agent: the model through the LLM gateway, the tools directly |
| D | What Highflame records |
| E | A rule that refuses one tool of that server |

[`strands_agentcore_id_token.ipynb`](strands_agentcore_id_token.ipynb) also sends the tool calls
through the Highflame MCP gateway. Use that path when the MCP server must be called with each
person's own connection.

## Setup

Do the one-time setup in the [README](README.md):
[Set it up in Highflame](README.md#set-it-up-in-highflame) (steps 1 to 5) and
[the agent's app](README.md#the-agents-app-both-flows) in Okta. Then:

1. **AWS Bedrock in Highflame.** In Studio → **AI Gateway** → **LLM Providers**, make sure that
   **AWS Bedrock** shows **Enabled**. It is on by default.
2. **AWS credentials** that may call the Bedrock model, for example after `aws sso login`.

The person must be a member of your Highflame organization, with the same email as in Okta.

This notebook uses the same `.env` as
[`strands_agentcore_id_token.ipynb`](strands_agentcore_id_token.ipynb). It does not use
`MCP_SERVER_URL`.

In [ ]:
%pip install -q -r requirements.txt

from importlib.metadata import version

from highflame.zeroid.resources.tokens import TokensResource

if not hasattr(TokensResource, "exchange_id_token"):
    raise RuntimeError(
        f"highflame {version('highflame')} has no tokens.exchange_id_token(). This notebook needs"
        " highflame 0.6.0 or later. Re-run the install cell above, then restart the kernel."
    )
for package in ["highflame", "strands-agents", "mcp"]:
    print(f"{package:<20} {version(package)}")

In [ ]:
import os

# common.py loads .env from this folder. Real environment variables win.
from common import agent_private_key, highflame_tokens, saved_id_token, scope, setting, show_claims
from signin import sign_in

for name in ["HIGHFLAME_IDENTITY_URL", "HIGHFLAME_LLM_URL", "AGENT_PRIVATE_KEY_PATH",
             "OKTA_ORG_URL", "OKTA_CLIENT_ID", "AWS_REGION"]:
    print(f"{name:<24} {'set' if os.environ.get(name) else 'MISSING: set it in .env'}")

import boto3

try:
    boto3.client("sts").get_caller_identity()
    print(f"{'AWS credentials':<24} found")
except Exception as e:  # noqa: BLE001 (any failure means: no usable credentials)
    print(f"{'AWS credentials':<24} MISSING ({type(e).__name__}): sign in to AWS, for example aws sso login")

## Part A. The person's token for the LLM gateway

The person signs in with Okta, as in the other notebook. The cell reuses the saved sign-in in
`.tokens/` while it is fresh. With no usable sign-in, it opens Okta's sign-in page: sign in **as the
person**, in a browser on this machine.

In [ ]:
id_token = saved_id_token() or sign_in()
if not id_token:
    raise RuntimeError("The person did not sign in. Run this cell again.")

show_claims("The person's ID token, from Okta", id_token, ["iss", "sub", "aud", "email"]);

The agent exchanges the ID token once, for a token bound (`resource`) to the LLM gateway. The
MCP server gets no Highflame token: the agent calls it directly.

In [ ]:
from highflame.zeroid import ZeroIDError

LLM_URL = setting("HIGHFLAME_LLM_URL")

try:
    llm_token = highflame_tokens().exchange_id_token(
        id_token,
        client_id=setting("OKTA_CLIENT_ID"),  # the ID token's aud, linked to the agent in Studio
        private_key_pem=agent_private_key(),   # private_key_jwt with the delegation key
        scope=scope(),
        resource=LLM_URL,                      # RFC 8707: this token works at the LLM gateway only
    ).access_token
except ZeroIDError as e:
    raise RuntimeError(f"Highflame refused the exchange ({e.status} {e.code}): {e.message}") from None

show_claims("The token for the LLM gateway", llm_token, ["sub", "principal_type", "act.sub", "aud"]);

## Part B. The MCP server, outside Highflame

The next cell starts [`example_mcp_server.py`](example_mcp_server.py) on this machine, at
`http://127.0.0.1:8099/mcp`, and connects to it directly. It is a made-up CRM with two tools:
`search_customers` returns names only, and `export_customers` returns every record with emails and
phone numbers.

In [ ]:
import subprocess
import sys
import time

import httpx
from strands.tools.mcp import MCPClient

CRM_URL = "http://127.0.0.1:8099/mcp"
crm_process = subprocess.Popen([sys.executable, "example_mcp_server.py"],
                               stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        httpx.get(CRM_URL, timeout=2)  # any HTTP answer means the server is up
        break
    except httpx.TransportError:
        time.sleep(1)
else:
    raise RuntimeError("The MCP server did not start. Run `python example_mcp_server.py` to see why.")

crm = MCPClient(url=CRM_URL)  # direct: Highflame is not on this path
crm.start()
crm_tools = crm.list_tools_sync()
print("tools of the MCP server:", [tool.tool_name for tool in crm_tools])

## Part C. The agent

The model is Bedrock **through the Highflame LLM gateway**: the short-lived Bedrock API key goes in
`Authorization`, and the person's token in `X-Highflame-Token`. The tools come from the MCP server
directly.

In [ ]:
from aws_bedrock_token_generator import provide_token
from strands import Agent
from strands.models.openai import OpenAIModel

REGION = os.environ.get("AWS_REGION", "us-east-1")
MODEL_ID = os.environ.get("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
QUESTION = "Export all customer records from the CRM, and tell me how many customers there are."


def ask(prompt: str) -> dict:
    """One request to the agent, for the person. Returns the tools that ran."""
    model = OpenAIModel(
        client_args={
            "base_url": LLM_URL,
            "api_key": provide_token(region=REGION),  # short-lived, from the AWS credentials
            "default_headers": {"X-Highflame-Token": llm_token},
        },
        model_id=f"bedrock/{MODEL_ID}",
        params={"max_tokens": 1024, "temperature": 0},
    )
    agent = Agent(model=model, tools=crm_tools, callback_handler=None,
                  system_prompt="You help with Acme's CRM. Use the CRM tools to answer.")
    result = agent(prompt)
    metrics = getattr(getattr(result, "metrics", None), "tool_metrics", {}) or {}
    ran = {name: getattr(m, "call_count", None) for name, m in metrics.items()}
    print(str(result).strip())
    print("\ntools that ran:", ran or "none")
    return ran


ask(QUESTION);

## Part D. What Highflame records

The MCP server answered the agent directly. Highflame still saw the tool call, in the model's
response. Open Studio → **Observatory** → **Traces**, and find the `llm.route` traces of the request
above:

| Span | What it holds |
| --- | --- |
| `llm.route` | One model call. `hf.user_id` is the person, `hf.agent_id` the agent. |
| `guard`, with `cedar.action` `call_tool` | The check of the tool call that the model asked for. `hf.tool_name` is `export_customers`, `hf.decision` is `allow`, and `hf.act_sub` is the person. |

The `guard` span has no `hf.mcp_server_name`: the LLM gateway knows the tool, not the server it
belongs to.

## Part E. Refuse one tool of that server

A gateway policy forbids `export_customers` by its name. It applies to every agent and every person
in the project, so try it in a test project.

Add it in Studio:

1. Go to **AI Gateway** → **Policies**, and select the project of your agent.
2. Open the **Tool Permissioning** tab, and choose **Create Policy**.
3. Give the policy a name. Set **Mode** to **Enforce**. A new policy starts in **Monitor**, which
   only records.
4. Choose **Add Rule**. Give the rule a name, and keep the effect **Forbid (Block)**.
5. In **Reject Message**, enter `Exporting customer records is not allowed.`
6. Under **Actions**, choose **Call Tool**.
7. Under **Conditions**, choose **Add Group**. Set the condition to **Tool Name**, **equals (==)**,
   `export_customers`.
8. Choose **Create Rule**, then **Deploy**.

The **Cedar** preview beside the rule shows the policy that Studio writes:

In [ ]:
print('''@name("Block export_customers")
@reject_message("Exporting customer records is not allowed.")
forbid (
  principal,
  action == AIGateway::Action::"call_tool",
  resource
)
when { context has tool_name && context.tool_name == "export_customers" };''')

The gateways apply a new policy within about 30 seconds. The next cell waits for it, then asks the
agent the same question again.

In [ ]:
time.sleep(35)
ran = ask(QUESTION)
if ran.get("export_customers"):
    print("\nThe tool still ran. Check that the policy is deployed, in Enforce mode, in the project of the agent.")

The LLM gateway refused the model's `export_customers` call before the agent could run it. The
agent reports the refusal, no tool ran, and the MCP server never got the call. In Observatory, the
`guard` span of that call has `hf.decision` `deny`.

When you are done, turn the policy's **Active** switch off, or delete it. Then stop the MCP server:

In [ ]:
crm.stop(None, None, None)
crm_process.terminate()
crm_process.wait(timeout=10)
print("MCP server stopped")

## Troubleshooting

See [Troubleshooting](README.md#troubleshooting) in the README for the sign-in and the exchange.

| What you see | Why |
| --- | --- |
| `unknown provider 'bedrock'` from the LLM gateway | AWS Bedrock is off under **AI Gateway** → **LLM Providers** (setup step 1). |
| The tool still runs after you deploy the policy | The policy is in **Monitor** mode, is not active, or is in another project. Or the gateways have not picked it up yet: wait 30 seconds. |
| The MCP server did not start | Port 8099 is in use. Set `EXAMPLE_MCP_PORT`, and change `CRM_URL` to match. |